In [ ]:
from google.colab import files
uploaded = files.upload()
# A file picker will appear — select both Fake.csv and True.csv

In [ ]:
import pandas as pd

# Load data
true = pd.read_csv("Fake.csv")
fake = pd.read_csv("True.csv")

# Add labels
fake['label'] = 0   # fake news
true['label'] = 1   # real news

# Combine
df = pd.concat([fake, true], ignore_index=True)

# Shuffle
df = df.sample(frac=1).reset_index(drop=True)

# Combine text columns (important for LSTM)
df['content'] = df['title'] + " " + df['text']

# Keep only needed columns
df = df[['content', 'label']]

# Drop null
df = df.dropna()

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)   # remove extra spaces
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    return text.strip()

df['content'] = df['content'].apply(clean_text)

df = df.drop_duplicates(subset='content')

# Check
print(df.head(5))
print(df['label'].value_counts())



In [ ]:
df.isnull().sum()


In [ ]:
X=df['content']
Y=df['label']

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42,shuffle=True,stratify=Y)




In [ ]:

from tensorflow.keras.preprocessing.text import Tokenizer
token = Tokenizer(num_words=20000, oov_token="<OOV>")
token.fit_on_texts(X_train)



In [ ]:
X_train_seq = token.texts_to_sequences(X_train)
X_test_seq = token.texts_to_sequences(X_test)


In [ ]:
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences
lengths = [len(seq) for seq in X_train_seq]
print("Max:", max(lengths))
print("Mean:", np.mean(lengths))
print("95 percentile:", np.percentile(lengths, 95))

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(X_train_seq, maxlen=1000, padding='pre')
X_test_pad = pad_sequences(X_test_seq, maxlen=1000, padding='pre')


In [ ]:
!pip install keras-tuner
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
def build_model(hp):
    model = Sequential()

    model.add(Embedding(
        input_dim=20000,
        output_dim=hp.Choice('embedding_dim', [64, 128, 256]),
    ))

    model.add(LSTM(
        units=hp.Choice('lstm_units', [32, 64, 128])
    ))

    model.add(Dropout(
        hp.Choice('dropout', [0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(
            learning_rate=hp.Choice('lr', [1e-3, 5e-4, 1e-4])
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory='tuner_utkaR',
    project_name='fake_news'
)


In [ ]:
tuner.search(
    X_train_pad,
    Y_train,
    epochs=3,
    validation_split=0.2
)

In [ ]:
#best_model = tuner.get_best_models(1)[0]
#best_model.summary()    DONE AND THE BEST PARAMS ARE embedding_dim = 64
#lstm_units = 128
#dropout = 0.3
#lr = 0.0005



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Embedding
model.add(Embedding(input_dim=20000, output_dim=64, input_length=540))

# LSTM
model.add(LSTM(128, dropout=0.3, recurrent_dropout=0.3))

# Dense
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

# Output
model.add(Dense(1, activation='sigmoid'))

# Compile
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0005),
    metrics=['accuracy']
)

In [ ]:
model = tuner.get_best_models(1)[0]

loss, acc = model.evaluate(X_test_pad, Y_test)
print("Test Accuracy:", acc)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Embedding
model.add(Embedding(input_dim=20000, output_dim=128, input_length=540))

# Single strong BiLSTM (faster + stable)
model.add(Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)))

# Dense layer (feature extraction)
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))

# Output
model.add(Dense(1, activation='sigmoid'))

# Compile
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0005),
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.optimizers import Adam, RMSprop

def build_model(hp):
    model = Sequential()

    model.add(Input(shape=(500,)))

    model.add(Embedding(
        input_dim=20000,
        output_dim=hp.Choice('embedding_dim', [64, 128, 256])
    ))

    model.add(Bidirectional(
        LSTM(hp.Choice('lstm_units', [32, 64, 128]))
    ))

    model.add(Dropout(
        hp.Choice('dropout', [0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation='sigmoid'))

    # 🔥 optimizer tuning
    optimizer_choice = hp.Choice('optimizer', ['adam', 'rmsprop'])

    lr = hp.Choice('lr', [1e-3, 5e-4, 1e-4])

    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=lr)
    else:
        optimizer = RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

In [ ]:
best_model = tuner.hypermodel.build(best_hp)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    Y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
model.evaluate(X_test_pad, Y_test)

In [ ]:
model.save("fake_news_model.h5")

import pickle
with open("token.pkl", "wb") as f:
    pickle.dump(token, f)

In [ ]:
import os
os.listdir('/kaggle/working')

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_news(text):
    text = clean_text(text)

    seq = token.texts_to_sequences([text])
    print("Sequence:", seq)   # DEBUG

    padded = pad_sequences(seq, maxlen=540)
    print("Padded sum:", padded.sum())  # DEBUG

    pred = model.predict(padded)[0][0]

    threshold = 0.6   # try 0.6 or 0.7

    if pred > threshold:
        print("Fake News")
    else:
        print("Real News")

In [ ]:
news = input("Enter news: ")
predict_news(news)

In [ ]:
import pickle

# model save
model.save("/kaggle/working/fake_news_model.h5")

# tokenizer save (tu ne 'token' use kiya tha)
with open("/kaggle/working/token.pkl", "wb") as f:
    pickle.dump(token, f)

In [ ]:
import os
os.listdir('/kaggle/working')